In [1]:
import numpy as np
import tensorflow as tf
import wave,math,string,os
from keras.models import Sequential
from tensorflow.keras.layers import Dense,LSTM,Activation

In [2]:
notes_freq={
    'A':440,'B':493.88,'C':261.63,'D':293.66,
    'E':393.63,'F':349.23,'G':392.00
}

In [3]:
notes_freq

{'A': 440,
 'B': 493.88,
 'C': 261.63,
 'D': 293.66,
 'E': 393.63,
 'F': 349.23,
 'G': 392.0}

In [4]:
notes=list(notes_freq.keys())

In [5]:
notes

['A', 'B', 'C', 'D', 'E', 'F', 'G']

In [6]:
note_to_int={note:i for i,note in enumerate(notes)}

In [7]:
note_to_int

{'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}

In [8]:
int_to_note={i:note for i,note in enumerate(notes)}

In [9]:
int_to_note

{0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F', 6: 'G'}

In [10]:
raw_music_data=[notes[np.random.randint(0,7)] for i in range(1000)]

In [11]:
raw_music_data

['D',
 'C',
 'G',
 'D',
 'F',
 'C',
 'E',
 'G',
 'D',
 'B',
 'B',
 'D',
 'A',
 'B',
 'E',
 'F',
 'D',
 'F',
 'E',
 'A',
 'G',
 'E',
 'D',
 'B',
 'D',
 'B',
 'D',
 'A',
 'A',
 'B',
 'B',
 'B',
 'G',
 'E',
 'D',
 'A',
 'C',
 'A',
 'C',
 'D',
 'C',
 'G',
 'D',
 'A',
 'G',
 'E',
 'E',
 'F',
 'A',
 'E',
 'G',
 'E',
 'C',
 'G',
 'A',
 'C',
 'G',
 'B',
 'C',
 'D',
 'E',
 'B',
 'A',
 'C',
 'F',
 'G',
 'C',
 'B',
 'E',
 'G',
 'D',
 'E',
 'B',
 'G',
 'E',
 'D',
 'A',
 'E',
 'G',
 'G',
 'F',
 'G',
 'F',
 'C',
 'F',
 'A',
 'A',
 'A',
 'F',
 'D',
 'G',
 'B',
 'F',
 'C',
 'A',
 'C',
 'G',
 'C',
 'B',
 'G',
 'A',
 'C',
 'G',
 'D',
 'E',
 'C',
 'F',
 'C',
 'A',
 'D',
 'E',
 'C',
 'C',
 'C',
 'G',
 'B',
 'F',
 'E',
 'D',
 'D',
 'B',
 'F',
 'B',
 'F',
 'F',
 'B',
 'G',
 'G',
 'C',
 'E',
 'F',
 'B',
 'D',
 'C',
 'B',
 'F',
 'B',
 'A',
 'A',
 'F',
 'E',
 'E',
 'D',
 'E',
 'G',
 'A',
 'E',
 'G',
 'B',
 'D',
 'E',
 'B',
 'G',
 'A',
 'B',
 'D',
 'G',
 'B',
 'D',
 'F',
 'E',
 'B',
 'B',
 'A',
 'B',
 'G',
 'F'

In [12]:
sequence_length = 3
network_input = []
network_output = []

for i in range(len(raw_music_data)-sequence_length):
    seq_in = raw_music_data[i: i+sequence_length]
    seq_out = raw_music_data[i+sequence_length]
    network_input.append([note_to_int[char] for char in seq_in])
    network_output.append(note_to_int[seq_out])

    print(seq_in, '--->', seq_out)

n_patterns = len(network_input)

['D', 'C', 'G'] ---> D
['C', 'G', 'D'] ---> F
['G', 'D', 'F'] ---> C
['D', 'F', 'C'] ---> E
['F', 'C', 'E'] ---> G
['C', 'E', 'G'] ---> D
['E', 'G', 'D'] ---> B
['G', 'D', 'B'] ---> B
['D', 'B', 'B'] ---> D
['B', 'B', 'D'] ---> A
['B', 'D', 'A'] ---> B
['D', 'A', 'B'] ---> E
['A', 'B', 'E'] ---> F
['B', 'E', 'F'] ---> D
['E', 'F', 'D'] ---> F
['F', 'D', 'F'] ---> E
['D', 'F', 'E'] ---> A
['F', 'E', 'A'] ---> G
['E', 'A', 'G'] ---> E
['A', 'G', 'E'] ---> D
['G', 'E', 'D'] ---> B
['E', 'D', 'B'] ---> D
['D', 'B', 'D'] ---> B
['B', 'D', 'B'] ---> D
['D', 'B', 'D'] ---> A
['B', 'D', 'A'] ---> A
['D', 'A', 'A'] ---> B
['A', 'A', 'B'] ---> B
['A', 'B', 'B'] ---> B
['B', 'B', 'B'] ---> G
['B', 'B', 'G'] ---> E
['B', 'G', 'E'] ---> D
['G', 'E', 'D'] ---> A
['E', 'D', 'A'] ---> C
['D', 'A', 'C'] ---> A
['A', 'C', 'A'] ---> C
['C', 'A', 'C'] ---> D
['A', 'C', 'D'] ---> C
['C', 'D', 'C'] ---> G
['D', 'C', 'G'] ---> D
['C', 'G', 'D'] ---> A
['G', 'D', 'A'] ---> G
['D', 'A', 'G'] ---> E
['A', 'G', 

In [13]:
x = np.reshape(network_input, (n_patterns, sequence_length, 1))

In [14]:
from keras.utils import to_categorical

In [15]:
y=to_categorical(network_output)
y.shape

(997, 7)

In [16]:
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(997, 7))

In [17]:
# Build the model
model = Sequential()
model.add(LSTM(256, input_shape=(x.shape[1], x.shape[2])))
model.add(Dense(1000, activation='relu'))
model.add(Dense(7, activation='softmax'))
model.summary()
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(x,y, epochs=100)

C:\Users\dai\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 256)            │       264,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1000)           │       257,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         7,007 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 528,199 (2.01 MB)

 Trainable params: 528,199 (2.01 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.1444 - loss: 1.9596
Epoch 2/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1655 - loss: 1.9463
Epoch 3/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1625 - loss: 1.9431
Epoch 4/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1675 - loss: 1.9426
Epoch 5/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1695 - loss: 1.9412
Epoch 6/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1464 - loss: 1.9399
Epoch 7/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1685 - loss: 1.9381
Epoch 8/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1625 - loss: 1.9403
Epoch 9/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1635 - loss: 1.9365
Epoch 10/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1735 - loss: 1.9364
Epoch 11/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1715 - loss: 1.9389
Epoch 12/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy:

In [18]:
start_index=np.random.randint(0,len(network_input))

In [19]:
pattern=network_input[start_index]

In [20]:
pattern

[2, 2, 3]

In [21]:
generated_melody=[]
    for i in range(16):
    x_input=np.reshape(pattern,(1,len(pattern),1))
    prediction=model.predict(x_input)
    index=np.argmax(prediction)
    result=int_to_note[index]
    generated_melody.append(result)
    pattern.append(index)
    pattern=pattern[1:len(pattern)]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step


In [22]:
pattern

[np.int64(3), np.int64(2), np.int64(6)]

In [23]:
generated_melody

['D',
 'D',
 'G',
 'D',
 'D',
 'E',
 'D',
 'D',
 'C',
 'G',
 'C',
 'F',
 'E',
 'D',
 'C',
 'G']

In [24]:
import struct

In [25]:
# Save this as audio file
with wave.open('my_music2.wav','w') as wav_file:
    wav_file.setparams((1,2,44100,0,'NONE','not compressed'))
    for note in generated_melody:
        freq=notes_freq[note]
        num_samples=int(0.5*44100) #duration*sample_rate
        for i in range(num_samples):
            #sample rate
            t=float(i)/44100
            value=int(32767*0.5*math.sin(2*math.pi*freq*t))
            data=struct.pack('<h',value)
            wav_file.writeframes(data)

In [26]:
help(wave.open)

Help on function open in module wave:

open(f, mode=None)

